<a href="https://arxiv.org/abs/1412.6980" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ADAM: A METHOD FOR STOCHASTIC OPTIMIZATION

This notebook implements the Adam optimizer as described in the paper **'ADAM: A METHOD FOR STOCHASTIC OPTIMIZATION'** by Diederik P. Kingma and Jimmy Ba. Adam combines the benefits of AdaGrad and RMSProp by computing adaptive learning rates for each parameter using estimates of the first and second moments of the gradients.

## Key Concepts:
- **First Moment Estimate (Mean)**: An exponentially decaying average of past gradients.
- **Second Moment Estimate (Uncentered Variance)**: An exponentially decaying average of past squared gradients.
- **Bias Correction**: To counteract the bias from initializing moments at zero.

### Parameter Update Rule:
$$ \theta_t = \theta_{t-1} - \alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} $$

Where:
- $ \hat{m}_t = \frac{m_t}{1 - \beta_1^t} $ (bias-corrected first moment)
- $ \hat{v}_t = \frac{v_t}{1 - \beta_2^t} $ (bias-corrected second moment)

This implementation uses a simple quadratic loss function for demonstration purposes.

In [ ]:
# Attempt to install torch if not present
try:
    import torch
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch'])
    import torch

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Define a simple quadratic loss function: L(x) = (x - 3)^2
# Minimum is at x = 3

class QuadraticLoss(nn.Module):
    def __init__(self):
        super(QuadraticLoss, self).__init__()
    
    def forward(self, x):
        return (x - 3) ** 2

loss_fn = QuadraticLoss()

In [ ]:
# Initialize parameter to optimize (1D tensor)
x = torch.tensor([0.0], requires_grad=True)
print(f"Initial parameter value: {x.item()}")

In [ ]:
# Adam hyperparameters
alpha = 0.1      # Learning rate
beta1 = 0.9      # Exponential decay rate for the first moment estimates
beta2 = 0.999    # Exponential decay rate for the second moment estimates
epsilon = 1e-8   # Small constant to prevent division by zero

In [ ]:
# Initialize first moment vector m0 = 0 and second moment vector v0 = 0
m = torch.zeros_like(x)
v = torch.zeros_like(x)
t = 0  # timestep

In [ ]:
# Compute gradient: gt = ∇θ ft(θt−1)
loss = loss_fn(x)
x_grad = torch.autograd.grad(loss, x)[0]
print(f"Gradient: {x_grad.item()}")

In [ ]:
# Update biased first moment estimate: mt = β1 · mt−1 + (1 − β1) · gt
m = beta1 * m + (1 - beta1) * x_grad

# Update biased second moment estimate: vt = β2 · vt−1 + (1 − β2) · gt²
v = beta2 * v + (1 - beta2) * (x_grad ** 2)

print(f"Updated first moment (biased): {m.item()}")
print(f"Updated second moment (biased): {v.item()}")

In [ ]:
# Compute bias-corrected first moment: bmt = mt / (1 - β1^t)
# Compute bias-corrected second moment: bvt = vt / (1 - β2^t)
t += 1
m_hat = m / (1 - beta1 ** t)
v_hat = v / (1 - beta2 ** t)

print(f"Bias-corrected first moment: {m_hat.item()}")
print(f"Bias-corrected second moment: {v_hat.item()}")

In [ ]:
# Update parameters: θt = θt−1 − α · bmt / (√bvt + ε)
with torch.no_grad():
    x -= alpha * m_hat / (torch.sqrt(v_hat) + epsilon)

print(f"Updated parameter: {x.item()}")

In [ ]:
# Reset states
x = torch.tensor([0.0], requires_grad=True)
m = torch.zeros_like(x)
v = torch.zeros_like(x)
t = 0

# Store values for visualization
losses = []
params = []

num_steps = 100
for step in range(num_steps):
    t += 1
    
    # Zero gradients
    if x.grad is not None:
        x.grad.zero_()
    
    # Forward pass
    loss = loss_fn(x)
    losses.append(loss.item())
    params.append(x.item())
    
    # Backward pass
    loss.backward()
    
    # Get gradient
    gt = x.grad
    
    # Update biased moments (eq. 1 & 2)
    m = beta1 * m + (1 - beta1) * gt
    v = beta2 * v + (1 - beta2) * (gt ** 2)
    
    # Bias correction (eq. 3)
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    
    # Parameter update (eq. 4)
    with torch.no_grad():
        x -= alpha * m_hat / (torch.sqrt(v_hat) + epsilon)

print(f"Final parameter after {num_steps} steps: {x.item()}")

In [ ]:
# Plotting convergence
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax[0].plot(losses)
ax[0].set_title("Loss vs Iteration")
ax[0].set_xlabel("Iteration")
ax[0].set_ylabel("Loss")

# Parameter convergence
ax[1].plot(params)
ax[1].axhline(y=3, color='r', linestyle='--', label='True Minimum')
ax[1].set_title("Parameter Value vs Iteration")
ax[1].set_xlabel("Iteration")
ax[1].set_ylabel("Parameter Value")
ax[1].legend()

plt.tight_layout()
plt.savefig("adam_convergence.png")

## Summary

In this notebook, we implemented the Adam optimizer from scratch using the equations provided in the paper. We applied it to a simple quadratic loss minimization problem and demonstrated its convergence properties.

Key steps implemented:
1. **Initialization** of first and second moment vectors ($m_0$, $v_0$).
2. **Gradient computation** using automatic differentiation.
3. **Moment updates** using exponential moving averages.
4. **Bias correction** to adjust for zero-initialized moments.
5. **Parameter update** using adaptive learning rates.

The visualization confirms that Adam efficiently converges towards the optimal solution. This implementation mirrors the behavior of `torch.optim.Adam` and helps understand how the algorithm works internally.